# NammaRetina — Improved DR Classifier (EfficientNetB0)

Target: **push past ~76% accuracy** on the APTOS-style 5-class split.

### What was wrong in the 76% notebook
1. Training used `train_ds_no_aug` — **augmentation was defined but never applied**
2. No **class weights** (APTOS is heavily imbalanced toward No DR)
3. Classification head was only `GAP → Dropout → Dense(5)` (too weak)
4. Checkpointed on accuracy only (DR contests care about **Quadratic Weighted Kappa**)

### What this notebook does
- Train **with** GPU augmentation + class weights + label smoothing
- Stronger head: `GAP → Dense(256) → BN → Dropout → Dense(5)`
- Phase 1: frozen backbone · Phase 2: fine-tune top ~40% layers
- Monitor **val QWK** + accuracy; export `efficientnetb0_dr.keras` for Render

### How to run
1. Runtime → Change runtime type → **T4 GPU**
2. Mount Drive, unzip `processed.zip`
3. Run all cells
4. Copy the exported model URL into Render env `MODEL_URL`


In [ ]:
# 0) Mount Drive + extract processed dataset
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
zip_path = '/content/drive/MyDrive/NammaRetina/processed.zip'
extract_path = '/content'
assert os.path.exists(zip_path), f'Missing {zip_path} — upload processed.zip to Drive first'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)
print('Train:', len(os.listdir('/content/processed/train_images')))
print('Val:', len(os.listdir('/content/processed/val_images')))
print('Test:', len(os.listdir('/content/processed/test_images')))


In [ ]:
# 1) Imports + config
import os, sys, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, GlobalAveragePooling2D, Dropout, BatchNormalization,
    RandomFlip, RandomRotation, RandomZoom, RandomContrast, RandomTranslation,
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, Callback
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')

gpus = tf.config.list_physical_devices('GPU')
print('TF', tf.__version__, '| GPUs:', gpus)
if not gpus:
    raise SystemExit('No GPU — switch Colab runtime to T4 GPU')
try:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
except Exception:
    pass

DATA_DIR = '/content/processed'
TRAIN_DIR = f'{DATA_DIR}/train_images'
VAL_DIR   = f'{DATA_DIR}/val_images'
TEST_DIR  = f'{DATA_DIR}/test_images'
TRAIN_CSV = f'{DATA_DIR}/train.csv'
VAL_CSV   = f'{DATA_DIR}/valid.csv'
TEST_CSV  = f'{DATA_DIR}/test.csv'

DRIVE_MODELS = '/content/drive/MyDrive/NammaRetina/models'
os.makedirs(DRIVE_MODELS, exist_ok=True)
LOCAL_MODELS = '/content/models'
os.makedirs(LOCAL_MODELS, exist_ok=True)

IMG_SIZE = 224
NUM_CLASSES = 5
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE
SEED = 42
tf.keras.utils.set_random_seed(SEED)

CLASS_NAMES = {0:'No DR', 1:'Mild', 2:'Moderate', 3:'Severe', 4:'Proliferative DR'}
BEST_MODEL_PATH = f'{LOCAL_MODELS}/efficientnetb0_dr.keras'
EXPORT_PATH = f'{DRIVE_MODELS}/efficientnetb0_dr.keras'

print('IMG_SIZE', IMG_SIZE, 'BATCH', BATCH_SIZE)
print('Will export →', EXPORT_PATH)


In [ ]:
# 2) Load CSVs + class weights
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)
for df in (train_df, val_df, test_df):
    df['filename'] = df['id_code'].astype(str) + '.png'

print(train_df['diagnosis'].value_counts().sort_index())

y = train_df['diagnosis'].values
cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y)
CLASS_WEIGHTS = {i: float(w) for i, w in enumerate(cw)}
print('Class weights:', CLASS_WEIGHTS)


In [ ]:
# 3) Augmentation (applied AFTER [-1,1] normalize — fill_value=0 is mid-grey)
augmentation_model = tf.keras.Sequential([
    RandomRotation(0.25, fill_mode='constant', fill_value=0.0),  # ±90° (safer than full 360)
    RandomFlip('horizontal'),
    RandomFlip('vertical'),
    RandomZoom((-0.12, 0.12), (-0.12, 0.12), fill_mode='constant', fill_value=0.0),
    RandomTranslation(0.08, 0.08, fill_mode='constant', fill_value=0.0),
    RandomContrast(0.20),
], name='retinal_augmentation')
print('Aug layers:', [l.name for l in augmentation_model.layers])


In [ ]:
# 4) tf.data pipeline
def load_and_preprocess(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.io.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 127.5 - 1.0  # MUST match model_loader.py
    return img, label


def create_dataset(dataframe, img_dir, shuffle=False, cache=True, augment=False):
    paths = [os.path.join(img_dir, f) for f in dataframe['filename'].values]
    one_hot = tf.keras.utils.to_categorical(dataframe['diagnosis'].values, NUM_CLASSES)
    ds = tf.data.Dataset.from_tensor_slices((paths, one_hot))
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    if cache:
        ds = ds.cache()
    if shuffle:
        ds = ds.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)
    if augment:
        ds = ds.map(lambda x, y: (augmentation_model(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

# CRITICAL: training uses augment=True
train_ds = create_dataset(train_df, TRAIN_DIR, shuffle=True,  cache=True, augment=True)
val_ds   = create_dataset(val_df,   VAL_DIR,   shuffle=False, cache=True, augment=False)
test_ds  = create_dataset(test_df,  TEST_DIR,  shuffle=False, cache=True, augment=False)
print('train/val/test batches ready (train WITH augmentation)')


In [ ]:
# 5) Build model — stronger head
base_model = EfficientNetB0(weights='imagenet', include_top=False,
                            input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_image')
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D(name='global_avg_pool')(x)
x = Dense(256, activation='relu', name='fc256')(x)
x = BatchNormalization(name='fc_bn')(x)
x = Dropout(0.40, name='fc_dropout')(x)
outputs = Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

model = Model(inputs, outputs, name='efficientnetb0_dr')
model.summary()


In [ ]:
# 6) QWK callback (primary DR metric)
class QuadraticKappaCallback(Callback):
    def __init__(self, validation_data, name='val_qwk'):
        super().__init__()
        self.validation_data = validation_data
        self.name = name
        self.best = -np.inf

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        y_true, y_pred = [], []
        for xb, yb in self.validation_data:
            probs = self.model.predict(xb, verbose=0)
            y_true.append(np.argmax(yb.numpy(), axis=1))
            y_pred.append(np.argmax(probs, axis=1))
        y_true = np.concatenate(y_true)
        y_pred = np.concatenate(y_pred)
        qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
        logs[self.name] = float(qwk)
        if qwk > self.best:
            self.best = qwk
        print(f' — {self.name}: {qwk:.4f} (best {self.best:.4f})')

qwk_cb = QuadraticKappaCallback(val_ds)


In [ ]:
# 7) Phase 1 — freeze backbone, train head
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ],
)

callbacks_p1 = [
    qwk_cb,
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(BEST_MODEL_PATH, monitor='val_accuracy', mode='max',
                    save_best_only=True, verbose=1),
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_p1,
    verbose=1,
)
print('Phase 1 done')


In [ ]:
# 8) Phase 2 — fine-tune top ~40% of EfficientNet
base_model.trainable = True
fine_tune_at = int(len(base_model.layers) * 0.60)  # freeze bottom 60%, train top 40%
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
print('Trainable backbone layers:', sum(1 for l in base_model.layers if l.trainable))

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss=CategoricalCrossentropy(label_smoothing=0.05),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ],
)

callbacks_p2 = [
    qwk_cb,
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint(BEST_MODEL_PATH, monitor='val_accuracy', mode='max',
                    save_best_only=True, verbose=1),
]

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_p2,
    verbose=1,
)
print('Phase 2 done')


In [ ]:
# 9) Evaluate best checkpoint on test set
best = tf.keras.models.load_model(BEST_MODEL_PATH)
results = best.evaluate(test_ds, verbose=1)
print({m: float(v) for m, v in zip(best.metrics_names, results)})

y_true, y_pred = [], []
for xb, yb in test_ds:
    probs = best.predict(xb, verbose=0)
    y_true.append(np.argmax(yb.numpy(), axis=1))
    y_pred.append(np.argmax(probs, axis=1))
y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)

print('\nQWK:', cohen_kappa_score(y_true, y_pred, weights='quadratic'))
print(classification_report(y_true, y_pred, target_names=[CLASS_NAMES[i] for i in range(5)]))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(5), yticklabels=range(5))
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Test Confusion Matrix')
plt.show()


In [ ]:
# 10) Export for deployment (Drive + local)
best.save(BEST_MODEL_PATH)
best.save(EXPORT_PATH)
print('Saved:', BEST_MODEL_PATH)
print('Saved:', EXPORT_PATH)
print()
print('Next for Render:')
print('1) Upload efficientnetb0_dr.keras to Google Drive (Anyone with link)')
print('2) Get a direct download link (or file id)')
print('3) Set Render env MODEL_URL=https://drive.google.com/uc?id=FILE_ID')
print('4) Redeploy — app downloads model on boot')
